In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings




warnings.filterwarnings("ignore")
pd.pandas.set_option("display.max_columns", None)
# Create Dataframe
df = pd.read_csv(r"./data/clustered_data.csv")
# Print shape of dataset
print(df.shape)

(2237, 22)


In [2]:
X = df.drop("cluster", axis=1) #dropping the target column which is 'cluster'
y = df["cluster"]

In [3]:

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report,ConfusionMatrixDisplay,precision_score, recall_score, f1_score, roc_auc_score,roc_curve,confusion_matrix
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn import metrics 

models = {
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "Logistic Regression": LogisticRegression(),
     "K-Neighbors Classifier": KNeighborsClassifier(),
    "XGBClassifier": XGBClassifier(), 
     "CatBoosting Classifier": CatBoostClassifier(verbose=False),
    "AdaBoost Classifier": AdaBoostClassifier()
}

In [4]:
from sklearn.model_selection import train_test_split
def evaluate_models(X, y, models):
    '''
    This function takes in X and y and models dictionary as input
    It splits the data into Train Test split
    Iterates through the given model dictionary and evaluates the metrics
    Returns: Dataframe which contains report of all models metrics with cost
    '''
    # separate dataset into train and test
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
    

    models_list = []
    scores = []
    
    for i in range(len(list(models))):
        model = list(models.values())[i]
        model.fit(X_train, y_train) # Train model

        # Make predictions
        y_pred = model.predict(X_test)

        score = accuracy_score(y_test,y_pred)
        
        model_name = list(models.keys())[i]
        print(f'---- score for --- {model_name} ----')
        print(f"{score}")
        models_list.append(model_name)
        scores.append(score)
    
    print()
    
    report = pd.DataFrame()
    report['Model_name'] = models_list
    report['Score'] = scores        
    return report

### Let's check the report

In [6]:
report = evaluate_models(X, y, models)

---- score for --- Random Forest ----
0.9821428571428571
---- score for --- Decision Tree ----
0.9776785714285714
---- score for --- Gradient Boosting ----
0.9888392857142857
---- score for --- Logistic Regression ----
0.9642857142857143
---- score for --- K-Neighbors Classifier ----
0.9375
---- score for --- XGBClassifier ----
0.984375
---- score for --- CatBoosting Classifier ----
0.984375
---- score for --- AdaBoost Classifier ----
0.9799107142857143



In [7]:
report.sort_values('Score')

,Model_name,Score
4,K-Neighbors Classifier,0.937500
3,Logistic Regression,0.964286
1,Decision Tree,0.977679
7,AdaBoost Classifier,0.979911
0,Random Forest,0.982143
5,XGBClassifier,0.984375
6,CatBoosting Classifier,0.984375
2,Gradient Boosting,0.988839


- ### From the report above we can see that the logistic regression model performed the best, so we will continue training our model using logistic regression algorithm.

In [9]:
# Separate data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.3)

X_train

,Age,Education,Marital Status,Parental Status,Children,Income,Total_Spending,Days_as_Customer,Recency,Wines,Fruits,Meat,Fish,Sweets,Gold,Web,Catalog,Store,Discount Purchases,Total Promo,NumWebVisitsMonth
1803,49,4,0,1,2,7144.0,416,4503,92,81,4,33,5.0,2,126.5,12,1,1.0,0,0,0
2021,50,4,1,1,1,71322.0,1305,4797,57,753,43,226,69.0,10,126.5,8,5,13.0,2,0,4
1949,64,2,0,0,0,80872.0,1336,4347,60,483,72,556,94.0,12,108.0,4,4,10.0,1,0,1
1591,48,4,0,1,1,52569.0,95,4384,54,85,0,3,0.0,0,7.0,2,0,4.0,1,0,3
1746,66,4,1,0,0,80360.0,2231,4782,56,1224,81,454,112.0,43,43.0,4,4,5.0,2,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1638,51,4,0,1,2,64140.0,1750,4578,71,1224,0,61,0.0,15,126.5,2,5,6.0,5,3,5
1095,33,3,1,1,1,27590.0,12,4731,38,6,0,5,0.0,0,1.0,1,0,2.0,1,0,7
1130,39,2,1,1,1,85606.0,1956,4893,89,717,42,556,120.5,30,84.0,6,7,9.0,2,1,3
1294,49,2,1,1,1,57811.0,802,4669,49,545,7,114,37.0,21,78.0,7,2,11.0,5,1,5


### Let's do hyperparameter tuning

In [ ]:
# Grid search cross validation
from sklearn.model_selection import GridSearchCV

params = {
    'solver': [ 'lbfgs', 'liblinear', 'sag', 'saga'],
    'max_iter': [150,250,500],
    'multi_class': [ 'ovr', 'multinomial'],
    'C': [0.01, 0.01, 0.1, 1,
    "penalty":["l1","l2"]

    
}

logreg=LogisticRegression()

logreg_cv=GridSearchCV(logreg,params,cv=10)
logreg_cv.fit(X_train,y_train)

print("tuned hpyerparameters :(best parameters) ",logreg_cv.best_params_)
print("accuracy :",logreg_cv.best_score_)

### So we got our best parameters. Let's now train the model with those parameters.

In [ ]:
best_lr_model = LogisticRegression(
    C = 1000, 
    max_iter =  113,
    multi_class =  'auto', 
    penalty =  'l2', 
    solver =  'lbfgs'
)

**Initialize model with best parameters**

### Let's check the report now

In [ ]:
best_model = best_lr_model.fit(X_train,y_train)
y_pred = best_model.predict(X_test)
score = accuracy_score(y_test,y_pred)
cr = classification_report(y_test,y_pred)

print("Logistic regression")
print ("Accuracy Score value: {:.4f}".format(score))
print (cr)

In [ ]:
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test)

- **Reports**

**We can see, that the model performed pretty well.**
- we have used logistic regression as it performed well than other models
- We got a good accuracy while predicting the test dataset.